In [1]:
import os
os.chdir("/home/ubuntu/work/dahyeon/backend")
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [6]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.modules.RAG.retriever import (
    full_hybrid,
    full_dense
)
import sys
from app.modules.RAG.anchor_book_pipeline import run_anchor_pipeline
from app.core.config import settings
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
nest_asyncio.apply()

In [7]:
REPO_ROOT     = Path("/home/ubuntu/work/dahyeon")
DATASET_DIR   = REPO_ROOT / "evaluation" / "dataset"
SCENARIO_PATH = DATASET_DIR / "scenario_data_v2.json"

with open(SCENARIO_PATH, encoding="utf-8") as f:
    scenarios = json.load(f)

def extract_rag_query(scenario: dict) -> dict:
    """마지막 turn의 rag_query를 꺼내고 query_id(=scenario_id)를 추가한다."""
    rag_query = scenario["turns"][-1]["rag_query"].copy()
    rag_query["query_id"] = scenario["scenario_id"]
    return rag_query

In [8]:
def simplify_item(item, source_name):
    return {
        "isbn":         item.get("isbn"),
        "title":        item.get("title"),
        "author":       item.get("author"),
        "publisher":    item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page":         item.get("page"),
        "large_cate":   item.get("large_cate"),
        "mid_cate":     item.get("mid_cate"),
        "small_cate":   item.get("small_cate"),
        "book_intro":   item.get("book_intro"),
        "book_index":   item.get("book_index"),
        "review_count": item.get("review_count"),
        "review_text":  item.get("review_text"),
    }


In [9]:
# ============================================================
# Strategy0 Dense 평가
# - index: books_review_full_100000
# - retrieval: full_dense
# - embedding model: KURE
# ============================================================

from pathlib import Path
import json
import pandas as pd
from tqdm import tqdm

INDEX_NAME = "books_review_full_100000"

EMBEDDING_MODEL = "kure"
EMBEDDING_FIELD = "embedding_kure"

TOP_K = 20
NUM_CANDIDATES = 300

OUTPUT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_eval.jsonl")

all_candidates = []

for scenario in tqdm(scenarios, desc="strategy0 dense experiment"):

    sample_result = extract_rag_query(scenario)

    if sample_result.get("anchor"):
        sample_result = run_anchor_pipeline(sample_result)

    query_id = sample_result["query_id"]

    re_dense = full_dense(
        result=sample_result,
        index_name=INDEX_NAME,
        size=TOP_K,
        num_candidates=NUM_CANDIDATES,
        embedding_field=EMBEDDING_FIELD,
        embedding_model=EMBEDDING_MODEL
    )

    for rank, book in enumerate(re_dense, start=1):
        all_candidates.append({
            "query_id": query_id,

            "strategy": "strategy0",
            "retrieval_type": "full_dense",
            "index_name": INDEX_NAME,

            "rank": rank,
            "score": book.get("score"),

            "dense_rank": book.get("dense_rank"),
            "dense_raw_score": book.get("dense_raw_score"),
            "has_dense": book.get("has_dense"),

            "isbn": str(book.get("isbn")),
            "title": book.get("title"),
            "author": book.get("author"),
            "publisher": book.get("publisher"),
            "publish_date": book.get("publish_date"),
            "page": book.get("page"),

            "large_cate": book.get("large_cate") or book.get("large"),
            "mid_cate": book.get("mid_cate"),
            "small_cate": book.get("small_cate"),

            "book_intro": book.get("book_intro"),
            "book_index": book.get("book_index"),
            "book_index_text": book.get("book_index_text"),
            "review": book.get("review"),
            "review_text": book.get("review_text"),
        })

print(f"\n전체 후보 도서: {len(all_candidates):,}")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for item in all_candidates:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"저장 완료: {OUTPUT_PATH}")

strategy0_dense_df = pd.DataFrame(all_candidates)
strategy0_dense_df.head()

strategy0 dense experiment: 100%|██████████| 42/42 [02:36<00:00,  3.73s/it]


전체 후보 도서: 840
저장 완료: /home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_eval.jsonl


,query_id,strategy,retrieval_type,index_name,rank,score,dense_rank,dense_raw_score,has_dense,isbn,...,publish_date,page,large_cate,mid_cate,small_cate,book_intro,book_index,book_index_text,review,review_text
0,S01,strategy0,full_dense,books_review_full_100000,1,0.803338,None,None,None,9791196797720,...,2020-03-27,216,[시/에세이],[나라별 에세이],[한국에세이],살다 보면 누구나 어쩔 수 없는 힘듦이 찾아온다. 어쩔 수 없는 힘듦은 마주한 힘듦...,"[마음처럼 안되는 순간을 만나, 불안하고 힘들다면, 나를 위한 시간, 불빛프로젝트,...",마음처럼 안되는 순간을 만나 불안하고 힘들다면 나를 위한 시간 불빛프로젝트 가능성 ...,{'reader_reaction': '생각보다 깊이가 없고 반복적인 내용으로 인해 ...,"생각보다 깊이가 없고 반복적인 내용으로 인해 실망감을 주지만, 일부 독자에게는 위로..."
1,S01,strategy0,full_dense,books_review_full_100000,2,0.802022,None,None,None,9791194006435,...,2024-12-18,180,[시/에세이],[],[],"일과 삶의 균형, 자존감 회복, 스트레스 관리 등 일반적으로 마주하는 다양한 고민과...","[Prologue, 1st 하루하루가 전쟁 같은 당신에게, 혼자여도 괜찮아, 나에게...",Prologue 1st 하루하루가 전쟁 같은 당신에게 혼자여도 괜찮아 나에게 건네는 위로,None,
2,S01,strategy0,full_dense,books_review_full_100000,3,0.800992,None,None,None,9791196539757,...,2025-03-10,232,"[시/에세이, 인문]",[나라별 에세이],[한국에세이],"“인간존재와 삶의 의미는 무엇인지, 왜 사는지, 무엇을 위해 어떻게 살 것인지, 늙...","[잠언 편, [ ], 삶의 본질에 대한 질문과 대답, 영적 세계와 앎에 대한 사유,...",잠언 편 [ ] 삶의 본질에 대한 질문과 대답 영적 세계와 앎에 대한 사유 삶의 방...,{'reader_reaction': '속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드...,속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드는 책입니다. 삶의 본질을 알고자 하거...
3,S01,strategy0,full_dense,books_review_full_100000,4,0.798929,None,None,None,9791186889329,...,2024-09-20,284,[시/에세이],"[나라별 에세이, 철학]","[철학에세이, 한국에세이]",세상을 건너다 종종 길을 잃는 그대에게\n노교수가 전하는 성찰의 메시지 68\n인간...,"[왜 살아야 하는지를 아는 사람, 남몰래 흐르는 눈물, 아무것도 하지 않기, 너는 ...",왜 살아야 하는지를 아는 사람 남몰래 흐르는 눈물 아무것도 하지 않기 너는 나의 자부심,"{'reader_reaction': '저자의 통찰력에 감탄했으며, 관계론적 존재론에...","저자의 통찰력에 감탄했으며, 관계론적 존재론에 대한 흥미로운 시각을 제시합니다. 저..."
4,S01,strategy0,full_dense,books_review_full_100000,5,0.796062,None,None,None,9791139201734,...,2021-11-19,264,[시/에세이],[나라별 에세이],[한국에세이],책 『진짜 어른이 된다는 것은』은 저자의 세 번째 에세이집으로 지난 한 해 동안 책...,"[출제 경향 분석 및 대비 방법, 국어, 핵심 정리 / 출제 유형 익히기 / 적중 ...",출제 경향 분석 및 대비 방법 국어 핵심 정리 / 출제 유형 익히기 / 적중 예상 ...,None,


In [14]:
from pathlib import Path
import sys
import json
import pandas as pd

# =====================================================
# 1. 파일 경로 / 설정
# =====================================================

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final_v2.json")
RETRIEVAL_RESULT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_eval.jsonl")

REPO_ROOT = Path("/home/ubuntu/work/dahyeon")
sys.path.insert(0, str(REPO_ROOT / "evaluation" / "metrics"))

from metrics import hit_at_k, recall_at_k, mrr_at_k, ndcg_at_k

K = 20
RELEVANCE_THRESHOLD = 2

# =====================================================
# 2. goldset 로드
# =====================================================

with GOLDSET_PATH.open("r", encoding="utf-8") as f:
    goldset = json.load(f)

gold_df = pd.DataFrame(goldset)

relevant_by_query = {}

for qid, g in gold_df.groupby("query_id"):
    relevant_isbns = (
        g[g["final_grade"] >= RELEVANCE_THRESHOLD]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )
    relevant_by_query[qid] = set(relevant_isbns)

print("전체 query 수:", len(relevant_by_query))
print("relevant 있는 query 수:", sum(1 for v in relevant_by_query.values() if v))

# =====================================================
# 3. 최종 Dense retrieval 결과 로드
# =====================================================

rows = []

with RETRIEVAL_RESULT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

retrieval_df = pd.DataFrame(rows)

print("retrieval 결과 수:", len(retrieval_df))
print("query 수:", retrieval_df["query_id"].nunique())

display(retrieval_df.head())

# =====================================================
# 4. query별 ranked isbn 구성
# =====================================================

ranked_by_query = {}

for qid, g in retrieval_df.groupby("query_id"):
    ranked = (
        g.sort_values("rank")["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )
    ranked_by_query[qid] = ranked

# =====================================================
# 5. 최종 Dense 평가
# =====================================================

metric_rows = []

for qid, rel in relevant_by_query.items():

    if not rel:
        continue

    ranked = ranked_by_query.get(qid, [])

    if not ranked:
        metric_rows.append({
            "query_id": qid,
            "k": K,
            "hit": 0,
            "recall": 0.0,
            "mrr": 0.0,
            "ndcg": 0.0,
            "goldset_count": len(rel),
            "retrieved_relevant_count": 0,
            "matched_isbns": [],
            "retrieved_count": 0,
        })
        continue

    topk = ranked[:K]
    hit_isbns = set(topk) & rel

    metric_rows.append({
        "query_id": qid,
        "k": K,
        "hit": hit_at_k(rel, ranked, K),
        "recall": recall_at_k(rel, ranked, K),
        "mrr": mrr_at_k(rel, ranked, K),
        "ndcg": ndcg_at_k(rel, ranked, K),
        "goldset_count": len(rel),
        "retrieved_relevant_count": len(hit_isbns),
        "matched_isbns": list(hit_isbns),
        "retrieved_count": len(topk),
    })

final_eval_df = pd.DataFrame(metric_rows)

print("평가 query 수:", len(final_eval_df))
display(final_eval_df.head())

# =====================================================
# 6. 최종 요약
# =====================================================

summary = {
    "strategy": "strategy0",
    "retrieval_type": "full_dense",
    "index_name": retrieval_df["index_name"].iloc[0] if "index_name" in retrieval_df.columns else None,
    "bm25_weight": retrieval_df["bm25_weight"].iloc[0] if "bm25_weight" in retrieval_df.columns else None,
    "dense_weight": retrieval_df["dense_weight"].iloc[0] if "dense_weight" in retrieval_df.columns else None,
    "k": K,
    "mean_hit": final_eval_df["hit"].mean(),
    "mean_recall": final_eval_df["recall"].mean(),
    "mean_mrr": final_eval_df["mrr"].mean(),
    "mean_ndcg": final_eval_df["ndcg"].mean(),
    "total_goldset_count": final_eval_df["goldset_count"].sum(),
    "total_retrieved_relevant_count": final_eval_df["retrieved_relevant_count"].sum(),
}

summary_df = pd.DataFrame([summary])

display(summary_df)

print("=" * 60)
print("최종 Dense 평가 결과")
print("=" * 60)
print("strategy:", summary["strategy"])
print("retrieval_type:", summary["retrieval_type"])
print("index_name:", summary["index_name"])
print("bm25_weight:", summary["bm25_weight"])
print("dense_weight:", summary["dense_weight"])
print(f"Hit@{K}:", round(summary["mean_hit"], 4))
print(f"Recall@{K}:", round(summary["mean_recall"], 4))
print(f"MRR@{K}:", round(summary["mean_mrr"], 4))
print(f"NDCG@{K}:", round(summary["mean_ndcg"], 4))
print("전체 relevant 수:", summary["total_goldset_count"])
print("검색된 relevant 수:", summary["total_retrieved_relevant_count"])
print("=" * 60)

전체 query 수: 39
relevant 있는 query 수: 39
retrieval 결과 수: 840
query 수: 42


,query_id,strategy,retrieval_type,index_name,rank,score,dense_rank,dense_raw_score,has_dense,isbn,...,publish_date,page,large_cate,mid_cate,small_cate,book_intro,book_index,book_index_text,review,review_text
0,S01,strategy0,full_dense,books_review_full_100000,1,0.803338,None,None,None,9791196797720,...,2020-03-27,216,[시/에세이],[나라별 에세이],[한국에세이],살다 보면 누구나 어쩔 수 없는 힘듦이 찾아온다. 어쩔 수 없는 힘듦은 마주한 힘듦...,"[마음처럼 안되는 순간을 만나, 불안하고 힘들다면, 나를 위한 시간, 불빛프로젝트,...",마음처럼 안되는 순간을 만나 불안하고 힘들다면 나를 위한 시간 불빛프로젝트 가능성 ...,{'reader_reaction': '생각보다 깊이가 없고 반복적인 내용으로 인해 ...,"생각보다 깊이가 없고 반복적인 내용으로 인해 실망감을 주지만, 일부 독자에게는 위로..."
1,S01,strategy0,full_dense,books_review_full_100000,2,0.802022,None,None,None,9791194006435,...,2024-12-18,180,[시/에세이],[],[],"일과 삶의 균형, 자존감 회복, 스트레스 관리 등 일반적으로 마주하는 다양한 고민과...","[Prologue, 1st 하루하루가 전쟁 같은 당신에게, 혼자여도 괜찮아, 나에게...",Prologue 1st 하루하루가 전쟁 같은 당신에게 혼자여도 괜찮아 나에게 건네는 위로,None,
2,S01,strategy0,full_dense,books_review_full_100000,3,0.800992,None,None,None,9791196539757,...,2025-03-10,232,"[시/에세이, 인문]",[나라별 에세이],[한국에세이],"“인간존재와 삶의 의미는 무엇인지, 왜 사는지, 무엇을 위해 어떻게 살 것인지, 늙...","[잠언 편, [ ], 삶의 본질에 대한 질문과 대답, 영적 세계와 앎에 대한 사유,...",잠언 편 [ ] 삶의 본질에 대한 질문과 대답 영적 세계와 앎에 대한 사유 삶의 방...,{'reader_reaction': '속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드...,속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드는 책입니다. 삶의 본질을 알고자 하거...
3,S01,strategy0,full_dense,books_review_full_100000,4,0.798929,None,None,None,9791186889329,...,2024-09-20,284,[시/에세이],"[나라별 에세이, 철학]","[철학에세이, 한국에세이]",세상을 건너다 종종 길을 잃는 그대에게\n노교수가 전하는 성찰의 메시지 68\n인간...,"[왜 살아야 하는지를 아는 사람, 남몰래 흐르는 눈물, 아무것도 하지 않기, 너는 ...",왜 살아야 하는지를 아는 사람 남몰래 흐르는 눈물 아무것도 하지 않기 너는 나의 자부심,"{'reader_reaction': '저자의 통찰력에 감탄했으며, 관계론적 존재론에...","저자의 통찰력에 감탄했으며, 관계론적 존재론에 대한 흥미로운 시각을 제시합니다. 저..."
4,S01,strategy0,full_dense,books_review_full_100000,5,0.796062,None,None,None,9791139201734,...,2021-11-19,264,[시/에세이],[나라별 에세이],[한국에세이],책 『진짜 어른이 된다는 것은』은 저자의 세 번째 에세이집으로 지난 한 해 동안 책...,"[출제 경향 분석 및 대비 방법, 국어, 핵심 정리 / 출제 유형 익히기 / 적중 ...",출제 경향 분석 및 대비 방법 국어 핵심 정리 / 출제 유형 익히기 / 적중 예상 ...,None,


평가 query 수: 39


,query_id,k,hit,recall,mrr,ndcg,goldset_count,retrieved_relevant_count,matched_isbns,retrieved_count
0,S01,20,1,0.222222,0.333333,0.218751,9,2,"[9791196539757, 9791186889329]",20
1,S03,20,1,0.484848,1.000000,0.826729,33,16,"[9791186821428, 9791170221920, 9791155814673, ...",20
2,S04,20,1,0.482759,1.000000,0.775363,29,14,"[9791168321175, 9788945078230, 9788958641766, ...",20
3,S05,20,1,0.333333,1.000000,0.515088,27,9,"[9791156029182, 9788985983945, 9788970270548, ...",20
4,S06,20,1,0.285714,1.000000,0.808828,56,16,"[9791193916636, 9791190764261, 9788973819676, ...",20


,strategy,retrieval_type,index_name,bm25_weight,dense_weight,k,mean_hit,mean_recall,mean_mrr,mean_ndcg,total_goldset_count,total_retrieved_relevant_count
0,strategy0,full_dense,books_review_full_100000,None,None,20,1.0,0.344564,0.801068,0.5729,1258,404


최종 Dense 평가 결과
strategy: strategy0
retrieval_type: full_dense
index_name: books_review_full_100000
bm25_weight: None
dense_weight: None
Hit@20: 1.0
Recall@20: 0.3446
MRR@20: 0.8011
NDCG@20: 0.5729
전체 relevant 수: 1258
검색된 relevant 수: 404


In [15]:
from pathlib import Path
import json

# =====================================================
# 파일 경로
# =====================================================

JSONL_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_eval.jsonl")
OUTPUT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_eval.json")

# =====================================================
# jsonl -> json 변환
# =====================================================

rows = []

with JSONL_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        rows.append(json.loads(line))

# =====================================================
# 저장
# =====================================================

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        rows,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 50)
print("변환 완료")
print("jsonl row 수:", len(rows))
print("저장 위치:", OUTPUT_PATH)
print("=" * 50)

변환 완료
jsonl row 수: 840
저장 위치: /home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_eval.json


# 정성적 평가 LLM-judge

In [16]:
import os
os.chdir("/home/ubuntu/work/dahyeon/backend")
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [17]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.core.config import settings
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

nest_asyncio.apply()

In [61]:
import json
import uuid
import time
import random
import asyncio
import requests
from pathlib import Path
from tqdm import tqdm

# =====================================================
# 0. 설정
# =====================================================

URL           = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
CLOVA_API_KEY = settings.CLOVA_API_KEY

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final_v2.json")
SCENARIO_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/scenario_data_v2.json")
RETRIEVAL_JSON_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_eval.json")
JSONL_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_llm_judge_v2.jsonl")
FINAL_JSON_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_llm_judge_v2.json")

scenario_map = {}
hard_negative_map = {}

TOP_K = 20
CONCURRENCY = 3
MAX_RETRIES = 5
LIMIT = None

EXCLUDE_KEYS = {
    "score",
    "bm25_rank",
    "dense_rank",
    "bm25_raw_score",
    "dense_raw_score",
    "bm25_rrf_score",
    "dense_rrf_score",
    "has_bm25",
    "has_dense",
}


# =====================================================
# 1. 시스템 프롬프트
# =====================================================

SYSTEM_PROMPT = """
당신은 도서 추천 검색 시스템(RAG 기반 Retrieval System)의 평가자입니다.

입력으로 사용자 검색 의도, RAG 쿼리 정보, 검색된 Top-20 도서 목록이 주어집니다.
당신의 역할은 검색 결과 전체가 사용자 의도와 조건을 얼마나 잘 만족하는지 평가하는 것입니다.

각 항목은 반드시 0점, 1점, 2점 중 하나로 채점하세요.

평가 항목은 다음 5개입니다.

1. result_diversity

- 모든 케이스에서 평가합니다.

- 다양성은 반드시 "쿼리의 핵심 의도를 유지한 범위 내에서" 평가합니다.

- 사용자가 특정 장르, 분위기, 시대, 대상층, 국가 등을 명시했다면,
  해당 축에 결과가 집중되는 것은 정상이며 diversity 감소 요인이 아닙니다.

- diversity는 핵심 의도를 벗어난 무관한 결과를 의미하지 않습니다.

- diversity는 동일한 핵심 조건을 만족하는 결과들 내부에서,
  소재, 세부 주제, 관계 구조, 분위기, 서사 방향, 직업/배경,
  성장 방식, 갈등 유형, 관점 등의 variation이 얼마나 존재하는지를 평가합니다.

- 2점:
  핵심 의도를 유지하면서도,
  최소 3개 이상의 하위 주제/관점/소재 variation이 균형 있게 존재합니다.

- 1점:
  핵심 의도는 유지하지만,
  일부 소재나 분위기에 편중되어 있습니다.
  다만 최소 2개 이상의 variation은 존재합니다.

- 0점:
  핵심 의도 내부에서도 대부분이 매우 유사한 소재/분위기/구조에 집중되어 있어,
  체감 가능한 variation이 거의 없습니다.

- 주의:
  쿼리 핵심 장르 자체에 집중되어 있다는 이유만으로 diversity를 낮게 평가하지 마세요.

2. result_focus
- 모든 케이스에서 평가합니다.
- 검색 결과가 특정 장르/주제에 얼마나 집중되어 있는지를 평가합니다.
- 결과 집합의 일관성과 노이즈 비율에 초점을 둡니다.

- 2점:
  검색 결과 대부분이 동일한 핵심 장르/주제 범위 안에 있습니다.
  노이즈가 적고 결과의 방향성이 일관됩니다.

- 1점:
  지정 장르/주제가 포함되어 있으나,
  관련성이 약한 결과나 다른 장르/주제가 함께 섞여 있습니다.

- 0점:
  절반 이상이 지정 장르/주제와 무관합니다.
  결과의 방향성이 크게 분산되어 있습니다.


3. query_alignment
- 모든 케이스에서 평가합니다.
- RAG 쿼리의 핵심 조건(장르, 분위기, 대상 독자, 난이도, 시대 배경, 소재 등)이 실제 검색 결과에 얼마나 반영되었는지를 평가합니다.
- 최종적으로 "사용 가능한 추천 후보군"이 얼마나 확보되었는지를 기준으로 판단합니다.

- 2점:
  최종 추천 후보군이 충분히 확보됨.
  (Top20 중 약 12~20권이 쿼리 의도와 핵심 조건에 적합)
  일부 노이즈는 허용됩니다.

- 1점:
  추천 가능한 책도 존재하지만,
  핵심 조건이 부분적으로만 반영되었거나 노이즈 비율이 높습니다.
  (Top20 중 약 6~11권 적합)

- 0점:
  추천 가능한 책이 거의 없으며,
  검색 결과가 쿼리 의도와 전반적으로 불일치합니다.
  (Top20 중 약 0~5권만 적합)


출력은 반드시 JSON만 반환하세요.
마크다운, 설명 문장, 코드블록은 출력하지 마세요.

출력 형식은 반드시 아래와 같습니다.

{
  "query_id": "",
  "scores": {
    "result_diversity": null,
    "result_focus": null,
    "query_alignment": 0
  },
  "total_score": 0,
  "max_score": 0,
  "pass": false,
  "reason": {
    "result_diversity": "",
    "result_focus": "",
    "query_alignment": ""
  }
}

규칙:
- total_score는 null이 아닌 항목 점수의 합입니다.
- max_score는 null이 아닌 항목 수 × 2입니다.
- pass는 total_score / max_score >= 0.7이면 true입니다.
- 반드시 유효한 JSON만 출력하세요.
"""


# =====================================================
# 2. 유틸 함수
# =====================================================

def make_headers():
    return {
        "Authorization": f"Bearer {CLOVA_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }


def parse_hcx_response(text: str) -> str:
    if "event:" in text:
        last_data = None

        for block in text.split("\n\n"):
            lines = block.strip().splitlines()
            event_type = None
            data_text = None

            for line in lines:
                if line.startswith("event:"):
                    event_type = line.replace("event:", "").strip()
                elif line.startswith("data:"):
                    data_text = line.replace("data:", "").strip()

            if event_type == "result" and data_text:
                last_data = data_text

        if last_data is not None:
            return json.loads(last_data)["message"]["content"]

    try:
        data = json.loads(text)

        if "result" in data:
            return data["result"]["message"]["content"]

        if "message" in data:
            return data["message"]["content"]

    except Exception:
        pass

    raise ValueError(f"응답 파싱 실패:\n{text[:1000]}")


def extract_json_object(llm_text: str) -> dict:
    """
    LLM이 혹시 앞뒤에 텍스트를 붙였을 경우 JSON 부분만 추출.
    """
    try:
        return json.loads(llm_text)
    except Exception:
        pass

    start = llm_text.find("{")
    end = llm_text.rfind("}")

    if start == -1 or end == -1 or start >= end:
        raise ValueError("JSON 객체를 찾지 못했습니다.")

    return json.loads(llm_text[start:end + 1])


def validate_judge_output(obj: dict) -> tuple:
    required_score_keys = [
        "result_diversity",
        "result_focus",
        "query_alignment"
    ]

    if "scores" not in obj:
        return None, "parse_failed", "missing scores"

    scores = obj["scores"]

    for key in required_score_keys:
        if key not in scores:
            return None, "parse_failed", f"missing score key: {key}"

        value = scores[key]

        if value is not None and value not in [0, 1, 2]:
            return None, "parse_failed", f"invalid score {key}: {value}"

    non_null_scores = [
        v for v in scores.values()
        if v is not None
    ]

    total_score = sum(non_null_scores)
    max_score = len(non_null_scores) * 2

    pass_label = False
    if max_score > 0:
        pass_label = (total_score / max_score) >= 0.7

    obj["total_score"] = total_score
    obj["max_score"] = max_score
    obj["pass"] = pass_label

    return obj, "success", None


def safe_str(value, max_len=700):
    if value is None:
        return ""

    text = str(value)

    if len(text) > max_len:
        return text[:max_len] + "..."

    return text


def make_book_for_judge(book: dict) -> dict:
    clean = {}

    for k, v in book.items():
        if k in EXCLUDE_KEYS:
            continue

        if str(k).startswith("retrieval_"):
            continue

        clean[k] = v

    return {
        "rank": clean.get("rank"),
        "isbn": clean.get("isbn"),
        "title": clean.get("title"),
        "author": clean.get("author"),
        "publisher": clean.get("publisher"),
        "publish_date": clean.get("publish_date"),
        "page": clean.get("page"),
        "large_cate": clean.get("large_cate"),
        "mid_cate": clean.get("mid_cate"),
        "small_cate": clean.get("small_cate"),
        "book_intro": safe_str(clean.get("book_intro"), 700),
        "book_index": safe_str(clean.get("book_index") or clean.get("book_index_text"), 700),
        "review": safe_str(clean.get("review_text") or clean.get("review"), 500),
    }


def build_judge_request(query_id: str, books: list) -> dict:
    first = books[0]

    scenario_info = scenario_map.get(query_id, {})
    rag_query = scenario_info.get("rag_query") or first.get("rag_query") or {}

    request_data = {
        "query_id": query_id,

        "query_type": scenario_info.get("query_type"),
        "scenario_type": scenario_info.get("scenario_type"),
        "query_specificity": scenario_info.get("query_specificity"),

        "final_user_input": scenario_info.get("final_user_input"),
        "turns": scenario_info.get("turns"),

        "keyword_query": rag_query.get("keyword_query"),
        "semantic_query": rag_query.get("semantic_query"),
        "filters": rag_query.get("filters"),
        "constraints": rag_query.get("constraints"),
        "score_boost": rag_query.get("score_boost"),
        "anchor": rag_query.get("anchor"),
        "session_signals": rag_query.get("session_signals"),
        "onboarding_signals": rag_query.get("onboarding_signals"),

        "diversity_mode": rag_query.get("diversity_mode"),
        "hard_negative_isbns": hard_negative_map.get(query_id, []),

        "preferred_categories": (
            scenario_info.get("onboarding", {}).get("preferred_categories")
            or rag_query.get("preferred_categories")
            or rag_query.get("onboarding_signals", {}).get("topic")
        ),

        "strategy": first.get("strategy"),
        "retrieval_type": first.get("retrieval_type"),
        "index_name": first.get("index_name"),
        "bm25_weight": first.get("bm25_weight"),
        "dense_weight": first.get("dense_weight"),
    }

    return {
        "request": request_data,
        "retrieved_books": [
            make_book_for_judge(book)
            for book in sorted(books, key=lambda x: x.get("rank", 999))[:TOP_K]
        ],
    }


# =====================================================
# 3. LLM judge 요청 함수
# =====================================================

def blocking_judge_one_query(query_id: str, books: list) -> dict:
    judge_input = build_judge_request(query_id, books)

    body = {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(judge_input, ensure_ascii=False, indent=2)
            },
        ],
        "topP": 0.1,
        "topK": 0,
        "max_tokens": 700,
        "temperature": 0.0,
        "repetitionPenalty": 1.0,
        "includeAiFilters": False,
        "seed": 42,
    }

    last_error = None

    for attempt in range(MAX_RETRIES):
        try:
            res = requests.post(
                URL,
                headers=make_headers(),
                json=body,
                timeout=90
            )

            if res.status_code == 200:
                llm_text = parse_hcx_response(res.text)
                obj = extract_json_object(llm_text)
                obj, label_status, parse_error = validate_judge_output(obj)

                if obj is not None:
                    obj["query_id"] = query_id

                return {
                    "query_id": query_id,
                    "judge_status": label_status,
                    "llm_raw_output": llm_text,
                    "llm_error": parse_error,
                    "judge_result": obj,
                    "judge_input": judge_input,
                }

            last_error = f"{res.status_code} / {res.text[:500]}"

            if res.status_code in [429, 500, 502, 503, 504]:
                raise Exception(last_error)

            return {
                "query_id": query_id,
                "judge_status": "api_failed",
                "llm_raw_output": None,
                "llm_error": last_error,
                "judge_result": None,
                "judge_input": judge_input,
            }

        except Exception as e:
            last_error = str(e)

            if attempt == MAX_RETRIES - 1:
                return {
                    "query_id": query_id,
                    "judge_status": "api_failed",
                    "llm_raw_output": None,
                    "llm_error": last_error,
                    "judge_result": None,
                    "judge_input": judge_input,
                }

            wait = min(30, (2 ** attempt) + random.uniform(0, 1))
            time.sleep(wait)


async def judge_one_query_async(query_id, books, semaphore, write_lock, jsonl_file, pbar):
    async with semaphore:
        try:
            result = await asyncio.to_thread(
                blocking_judge_one_query,
                query_id,
                books
            )
        except Exception as e:
            result = {
                "query_id": query_id,
                "judge_status": "api_failed",
                "llm_raw_output": None,
                "llm_error": str(e),
                "judge_result": None,
                "judge_input": None,
            }

    async with write_lock:
        jsonl_file.write(json.dumps(result, ensure_ascii=False) + "\n")
        jsonl_file.flush()

    pbar.update(1)

    return result


# =====================================================
# 4. main
# =====================================================

async def main():
    global scenario_map, hard_negative_map

    with SCENARIO_PATH.open("r", encoding="utf-8") as f:
        scenarios = json.load(f)

    scenario_map = {}

    for s in scenarios:
        qid = s.get("scenario_id") or s.get("query_id")

        final_turn = None
        for turn in s.get("turns", []):
            if turn.get("rag_query"):
                final_turn = turn

        if final_turn is None:
            continue

        scenario_map[qid] = {
            "scenario_id": qid,
            "query_type": s.get("query_type"),
            "scenario_type": s.get("scenario_type"),
            "query_specificity": s.get("query_specificity"),
            "onboarding": s.get("onboarding"),
            "turns": s.get("turns"),
            "final_user_input": final_turn.get("user_input"),
            "rag_query": final_turn.get("rag_query"),
        }

    print("scenario rag_query 수:", len(scenario_map))
    
    with GOLDSET_PATH.open("r", encoding="utf-8") as f:
        goldset = json.load(f)

    hard_negative_map = {}

    for item in goldset:
        qid = item.get("query_id")
        isbn = item.get("isbn")
        final_grade = item.get("final_grade")

        if qid is None or isbn is None:
            continue

        if final_grade == 1:
            hard_negative_map.setdefault(qid, set()).add(str(isbn))

    hard_negative_map = {
        qid: sorted(list(isbns))
        for qid, isbns in hard_negative_map.items()
    }

    print("hard negative 있는 query 수:", len(hard_negative_map))

    with RETRIEVAL_JSON_PATH.open("r", encoding="utf-8") as f:
        candidates = json.load(f)

    grouped = {}

    for book in candidates:
        qid = book.get("query_id")

        if not qid:
            continue

        grouped.setdefault(qid, []).append(book)

    print(f"전체 후보 도서 수: {len(candidates):,}")
    print(f"전체 query 수: {len(grouped):,}")

    processed_query_ids = set()
    already_judged = []

    if JSONL_PATH.exists():
        with JSONL_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                try:
                    item = json.loads(line)
                    qid = item.get("query_id")

                    if qid and qid not in processed_query_ids:
                        processed_query_ids.add(qid)
                        already_judged.append(item)

                except json.JSONDecodeError:
                    continue

        print(f"이미 처리된 query 수: {len(processed_query_ids):,}")

    to_process = [
        (qid, books)
        for qid, books in grouped.items()
        if qid not in processed_query_ids
    ]

    if LIMIT is not None:
        to_process = to_process[:LIMIT]

    print(f"처리 대상 query 수: {len(to_process):,}")

    if not to_process:
        print("처리할 query가 없습니다.")
        return already_judged

    semaphore = asyncio.Semaphore(CONCURRENCY)
    write_lock = asyncio.Lock()
    pbar = tqdm(total=len(to_process), desc="LLM judge 중", unit="query")

    with JSONL_PATH.open("a", encoding="utf-8") as jsonl_file:
        tasks = [
            judge_one_query_async(
                qid,
                books,
                semaphore,
                write_lock,
                jsonl_file,
                pbar
            )
            for qid, books in to_process
        ]

        new_results = await asyncio.gather(*tasks)

    pbar.close()

    all_results = already_judged + list(new_results)

    query_order = {
        qid: i
        for i, qid in enumerate(grouped.keys())
    }

    all_results.sort(
        key=lambda x: query_order.get(x.get("query_id"), 999999)
    )

    with FINAL_JSON_PATH.open("w", encoding="utf-8") as f:
        json.dump(
            all_results,
            f,
            ensure_ascii=False,
            indent=2
        )

    total_success = sum(
        1 for r in all_results
        if r.get("judge_status") == "success"
    )

    total_failed = len(all_results) - total_success

    print("=" * 60)
    print("LLM judge 완료")
    print("총 query 수:", len(all_results))
    print("성공:", total_success)
    print("실패:", total_failed)
    print("JSONL 저장:", JSONL_PATH)
    print("JSON 저장:", FINAL_JSON_PATH)
    print("=" * 60)

    return all_results


results = await main()

scenario rag_query 수: 42
hard negative 있는 query 수: 39
전체 후보 도서 수: 840
전체 query 수: 42
처리 대상 query 수: 42


LLM judge 중: 100%|██████████| 42/42 [02:47<00:00,  4.00s/query]

LLM judge 완료
총 query 수: 42
성공: 42
실패: 0
JSONL 저장: /home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_llm_judge_v2.jsonl
JSON 저장: /home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_llm_judge_v2.json


In [62]:
import json
import pandas as pd
from pathlib import Path

# =====================================================
# 파일 경로
# =====================================================

JUDGE_PATH = Path(
    "/home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_llm_judge_v2.json"
)

# =====================================================
# 로드
# =====================================================

with JUDGE_PATH.open("r", encoding="utf-8") as f:
    judge_results = json.load(f)

print("전체 query 수:", len(judge_results))

# =====================================================
# query별 점수 정리
# =====================================================

rows = []

for item in judge_results:

    query_id = item.get("query_id")
    judge_status = item.get("judge_status")

    judge_result = item.get("judge_result") or {}
    scores = judge_result.get("scores") or {}
    reasons = judge_result.get("reason") or {}

    rows.append({
        "query_id": query_id,
        "judge_status": judge_status,

        "result_diversity": scores.get("result_diversity"),
        "result_focus": scores.get("result_focus"),
        "query_alignment": scores.get("query_alignment"),
        "hard_negative_exclusion": scores.get("hard_negative_exclusion"),
        "category_fit": scores.get("category_fit"),

        "total_score": judge_result.get("total_score"),
        "max_score": judge_result.get("max_score"),
        "pass": judge_result.get("pass"),

        "reason_query_alignment": reasons.get("query_alignment"),
        "reason_category_fit": reasons.get("category_fit"),
    })

judge_df = pd.DataFrame(rows)

# =====================================================
# 점수 높은 순 정렬
# =====================================================

judge_df = judge_df.sort_values(
    ["total_score", "query_id"],
    ascending=[False, True]
)

# =====================================================
# 출력
# =====================================================

pd.set_option("display.max_colwidth", 300)

#display(judge_df)

# =====================================================
# 평균 점수 요약
# =====================================================

judge_df = judge_df.sort_values(
    by="query_id",
    key=lambda x: x.str.extract(r"(\d+)").astype(int)[0]
).reset_index(drop=True)

summary = {
    "query_count": len(judge_df),
    "pass_count": int(judge_df["pass"].sum()),
    "pass_ratio": round(judge_df["pass"].mean(), 4),

    "avg_total_score": round(judge_df["total_score"].mean(), 4),

    "avg_result_diversity":
        round(judge_df["result_diversity"].dropna().mean(), 4),

    "avg_result_focus":
        round(judge_df["result_focus"].dropna().mean(), 4),

    "avg_query_alignment":
        round(judge_df["query_alignment"].dropna().mean(), 4),

    "avg_hard_negative_exclusion":
        round(judge_df["hard_negative_exclusion"].dropna().mean(), 4),

    "avg_category_fit":
        round(judge_df["category_fit"].dropna().mean(), 4),
}

summary_df = pd.DataFrame([summary])

display(judge_df)
display(summary_df)

전체 query 수: 42


,query_id,judge_status,result_diversity,result_focus,query_alignment,hard_negative_exclusion,category_fit,total_score,max_score,pass,reason_query_alignment,reason_category_fit
0,S01,success,2,2,2,None,None,6,6,True,사용자의 '실존주의적 느낌의 에세이' 요청에 부합하는 책들이 다수 포함됨,None
1,S02,success,1,2,1,None,None,4,6,False,"장하준 스타일의 책 구체적 부족, 일부 책은 '실무 적용' 요구와 완벽히 일치하지 않음",None
2,S03,success,2,2,2,None,None,6,6,True,"쿼리의 핵심 조건(우주 관련, 2015년 이후, 400페이지 이하, 이야기 형식)에 부합하는 책들이 다수 포함되어 있습니다.",None
3,S04,success,2,2,2,None,None,6,6,True,대부분의 검색 결과가 '팀장이 된 중급 독자를 위한 실무 리더십·팀 관리 책'이라는 사용자의 요청에 부합합니다.,None
4,S05,success,2,2,2,None,None,6,6,True,"쿼리의 핵심 주제인 '노년기 삶에 대한 인문 에세이'와 제약 조건인 '한국 작가', '250페이지 이하', '에세이 형식'이 잘 반영됨",None
5,S06,success,2,2,2,None,None,6,6,True,쿼리의 핵심 조건인 '인간관계가 힘들 때 위로와 공감이 되는 자기계발·에세이'에 부합하는 책들이 충분히 확보됨,None
6,S07,success,2,2,2,None,None,6,6,True,"검색 결과 중 많은 책들이 주식과 ETF 기초에 관한 내용을 포함하고 있으며, 사용자의 조건(2020년 이후, 300페이지 이하, 학문적이지 않은 스타일)에도 부합합니다.",None
7,S08,success,2,2,2,None,None,6,6,True,"세계사/한국사 배경, 청소년 독자층, 400페이지 이하 분량 등 핵심 조건이 잘 반영되었습니다.",None
8,S09,success,2,1,1,None,None,4,6,False,"쿼리 요구에 부분적으로 부합하는 책들이 있으나, 노이즈 비율이 높아 쿼리와의 일치성이 부분적으로만 충족됩니다.",None
9,S10,success,2,2,2,None,None,6,6,True,검색 결과 중 대부분이 '나이 들면서 어떻게 살아야 하나'라는 주제와 직접적으로 관련된 에세이들입니다.,None


,query_count,pass_count,pass_ratio,avg_total_score,avg_result_diversity,avg_result_focus,avg_query_alignment,avg_hard_negative_exclusion,avg_category_fit
0,42,27,0.6429,4.9524,1.619,1.6667,1.6667,NaN,NaN


In [20]:
from pathlib import Path
import json
import pandas as pd

# =====================================================
# 파일 경로
# =====================================================

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final_v2.json")
RETRIEVAL_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_eval.json")

TOP_K = 20

# =====================================================
# 1. goldset 로드
# =====================================================

with GOLDSET_PATH.open("r", encoding="utf-8") as f:
    goldset = json.load(f)

gold_df = pd.DataFrame(goldset)

# =====================================================
# 2. hard negative map 생성
#    final_grade == 1
# =====================================================

hard_negative_map = {}

for qid, g in gold_df.groupby("query_id"):

    hard_isbns = (
        g[g["final_grade"] == 1]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    hard_negative_map[qid] = set(hard_isbns)

print("hard negative query 수:", len(hard_negative_map))

# =====================================================
# 3. retrieval 결과 로드
# =====================================================

with RETRIEVAL_PATH.open("r", encoding="utf-8") as f:
    retrieval = json.load(f)

retrieval_df = pd.DataFrame(retrieval)

print("retrieval row 수:", len(retrieval_df))

# =====================================================
# 4. query별 hard negative 포함 여부 계산
# =====================================================

rows = []

for qid, g in retrieval_df.groupby("query_id"):

    hard_negatives = hard_negative_map.get(qid, set())

    if not hard_negatives:
        continue

    g = g.sort_values("rank")

    top20 = (
        g[g["rank"] <= 20]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    top10 = (
        g[g["rank"] <= 10]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    rank11_20 = (
        g[(g["rank"] >= 11) & (g["rank"] <= 20)]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    top20_hard = sorted(list(set(top20) & hard_negatives))
    top10_hard = sorted(list(set(top10) & hard_negatives))
    rank11_20_hard = sorted(list(set(rank11_20) & hard_negatives))

    rows.append({
        "query_id": qid,

        "hard_negative_total": len(hard_negatives),

        "hard_in_top20_count": len(top20_hard),
        "hard_in_top10_count": len(top10_hard),
        "hard_in_11_20_count": len(rank11_20_hard),

        "hard_in_top20": top20_hard,
        "hard_in_top10": top10_hard,
        "hard_in_11_20": rank11_20_hard,
    })

result_df = pd.DataFrame(rows)

# =====================================================
# 5. 전체 통계
# =====================================================

print("=" * 60)

print("Top20 안에 hard negative 있는 query 수:")
print((result_df["hard_in_top20_count"] > 0).sum())

print()

print("Top10 안에 hard negative 있는 query 수:")
print((result_df["hard_in_top10_count"] > 0).sum())

print()

print("11~20위에만 hard negative 있는 query 수:")
print(
    (
        (result_df["hard_in_top10_count"] == 0)
        &
        (result_df["hard_in_11_20_count"] > 0)
    ).sum()
)

print("=" * 60)

# =====================================================
# 6. 상세 확인
# =====================================================

display(
    result_df.sort_values(
        ["query_id", "hard_in_top10_count", "hard_in_11_20_count"],
        ascending=[True, False, False],
        key=lambda col: (
            col.str.extract(r"(\d+)").astype(int)[0]
            if col.name == "query_id"
            else col
        )
    ).reset_index(drop=True)
)

hard negative query 수: 39
retrieval row 수: 840
Top20 안에 hard negative 있는 query 수:
38

Top10 안에 hard negative 있는 query 수:
38

11~20위에만 hard negative 있는 query 수:
0


,query_id,hard_negative_total,hard_in_top20_count,hard_in_top10_count,hard_in_11_20_count,hard_in_top20,hard_in_top10,hard_in_11_20
0,S01,69,13,8,5,"[9791139201734, 9791139212457, 9791141061876, 9791141903374, 9791141996772, 9791163383031, 9791164050840, 9791192942865, 9791194006435, 9791194299202, 9791194648079, 9791196797720, 9791196956905]","[9791139201734, 9791141061876, 9791141903374, 9791141996772, 9791192942865, 9791194006435, 9791196797720, 9791196956905]","[9791139212457, 9791163383031, 9791164050840, 9791194299202, 9791194648079]"
1,S03,43,4,1,3,"[9788958202899, 9791157846313, 9791193778036, 9791195686858]",[9791157846313],"[9788958202899, 9791193778036, 9791195686858]"
2,S04,50,6,2,4,"[9788955960563, 9788958612407, 9788959890316, 9788972574699, 9788993111071, 9788993634112]","[9788959890316, 9788972574699]","[9788955960563, 9788958612407, 9788993111071, 9788993634112]"
3,S05,45,7,1,6,"[9788972676027, 9788997483495, 9791157782413, 9791162757017, 9791189930295, 9791191013917, 9791198898845]",[9788972676027],"[9788997483495, 9791157782413, 9791162757017, 9791189930295, 9791191013917, 9791198898845]"
4,S06,23,4,2,2,"[9788975070648, 9788991984509, 9791141037857, 9791160544152]","[9788975070648, 9791141037857]","[9788991984509, 9791160544152]"
5,S07,39,5,1,4,"[9788947500746, 9788998342784, 9791141088521, 9791170432821, 9791192942254]",[9791192942254],"[9788947500746, 9788998342784, 9791141088521, 9791170432821]"
6,S08,50,9,5,4,"[9788932404882, 9788955520026, 9788972531296, 9788978150125, 9788988295106, 9788989721321, 9788992673082, 9788993192001, 9788993379143]","[9788972531296, 9788988295106, 9788992673082, 9788993192001, 9788993379143]","[9788932404882, 9788955520026, 9788978150125, 9788989721321]"
7,S09,48,14,8,6,"[9788932006673, 9788936436889, 9788936438005, 9788936616083, 9788952770394, 9788960781672, 9788979197570, 9788995655245, 9791137296541, 9791156290230, 9791189217358, 9791190492829, 9791193710975, 9791197089916]","[9788932006673, 9788936436889, 9788936616083, 9788979197570, 9791137296541, 9791156290230, 9791189217358, 9791190492829]","[9788936438005, 9788952770394, 9788960781672, 9788995655245, 9791193710975, 9791197089916]"
8,S10,21,3,1,2,"[9788964200452, 9791164160037, 9791190162524]",[9788964200452],"[9791164160037, 9791190162524]"
9,S11,61,15,7,8,"[9788932006673, 9788932015606, 9788936616083, 9788941329251, 9788954611275, 9788977182714, 9788980502042, 9788990707635, 9791138833530, 9791141966881, 9791156290230, 9791170320890, 9791191797169, 9791192265599, 9791194595731]","[9788932006673, 9788936616083, 9788941329251, 9788980502042, 9791138833530, 9791141966881, 9791156290230]","[9788932015606, 9788954611275, 9788977182714, 9788990707635, 9791170320890, 9791191797169, 9791192265599, 9791194595731]"


# Human 정성적평가

In [64]:
from pathlib import Path
import json
import pandas as pd

RETRIEVAL_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_dense_eval.json")

with RETRIEVAL_PATH.open("r", encoding="utf-8") as f:
    retrieval = json.load(f)

df = pd.DataFrame(retrieval)

def short_text(x, n=180):
    if x is None:
        return ""
    return str(x).replace("\n", " ")[:n]

def show_query_results(qid):
    qdf = (
        df[df["query_id"] == qid]
        .sort_values("rank")
        .head(20)
        .copy()
    )

    if qdf.empty:
        print(f"{qid} 결과 없음")
        return

    qdf["book_intro_short"] = qdf["book_intro"].apply(lambda x: short_text(x, 180))

    view_cols = [
        "rank",
        "isbn",
        "title",
        "author",
        "large_cate",
        "mid_cate",
        "small_cate",
        "page",
        "publish_date",
        "book_intro_short",
    ]

    print("=" * 100)
    print(f"QUERY: {qid}")
    print("=" * 100)

    print("\n[large_cate 분포]")
    display(qdf["large_cate"].explode().value_counts().reset_index())

    print("\n[mid_cate 분포]")
    display(qdf["mid_cate"].explode().value_counts().reset_index())

    print("\n[Top-20 결과]")
    display(qdf[view_cols].reset_index(drop=True))


# 예시
show_query_results("S27")

QUERY: S27

[large_cate 분포]


,large_cate,count
0,시/에세이,19
1,예술/대중문화,3



[mid_cate 분포]


,mid_cate,count
0,나라별 에세이,14
1,테마에세이,6
2,미술,3
3,디자인/색채,1



[Top-20 결과]


,rank,isbn,title,author,large_cate,mid_cate,small_cate,page,publish_date,book_intro_short
0,1,9791195513062,하루가 미안해서 - 사소해서 더 아름다운 삶의 작은 조각들,김학수 저,[시/에세이],[나라별 에세이],[한국에세이],192,2018-06-20,"사소해서 더 아름다운 삶의 작은 조각들 소소한 일상을 작은 발견으로 그려낸 그림 에세이 『하루가 미안해서』는 우리 주변의 작고 사소한 것들을 담담하게 담아낸 일러스트 에세이다. 일과 사랑, 커피, 산책 등 우리의 일상에서 스쳐 지나갈 듯한 소소한 이야기들을 작은 발견으로 그려냈다. 텀블벅 크라우드 펀딩을 통해 먼저 소개되"
1,2,9788993200034,끄덕끄덕,신영준 저,[시/에세이],"[나라별 에세이, 테마에세이]","[포토에세이, 한국에세이]",203,2014-07-20,일상의 평범함에서 찾는 특별한 의미를 담고 있다.
2,3,9791197151866,당신을 보면 이해받는 기분이 들어요 - 공간 시리즈 3,"김건희, 김지연 저","[시/에세이, 예술/대중문화]","[나라별 에세이, 테마에세이]","[일기/편지, 한국에세이]",204,2023-07-10,"미술관과 갤러리는 작품을 위한 곳인 동시에 그곳을 드나드는 많은 사람의 삶이 담긴 공간이다. 예술을 매개로 10살 차이를 넘어 가까운 친구가 된 두 여성이 전시공간에서 경험한 다양한 이야기, 예술에 대한 생각을 계절의 흐름이 느껴지는 편지 형식의 에세이로 나눈다. 아름다움이란 무엇인지, 예술은 우리 삶에 어떤 의미를 주는"
3,4,9791141031268,일상을 바꿔주는 감사 일기,이예지 저,[시/에세이],[],[],103,2023-06-09,평범한 일상 속에서의 사소한 감사 거리를 찾아 감사하는 저자의 짧은 에세이이다.
4,5,9788963394152,느리게 걷다 당신을 만나다,임정일 저,[시/에세이],[나라별 에세이],[한국에세이],240,2014-11-10,"'느리게 걷다, 당신을 만나다'는 매일 스치듯 흘려보낸 풍경들을 찬찬히 둘러보며 일상의 소소한 행복을 일깨워준다. 우리는 모두 행복해질 수 있는 권리를 갖고 있음을 강조하며 삶의 용기와 사랑을 속삭여준다. 잊혀가는 것들을 되짚으며 여유 있는 마음을 가질 때야 비로소 행복에 근접해질 수 있다."
5,6,9788996264033,에브리띵 이즈 썸띵 투 어스 EVERYTHING IS SOMETHING TO US,(주)밀리미터밀리그람 저,[시/에세이],[디자인/색채],[디자인],108,2010-06-18,일상적이고 사소한 것들이 가진 특별함을 이야기하는 포토 에세이 『EVERYTHING IS SOMETHING TO US』. 디자인 소품 브랜드 MMMG만의 섬세한 감각을 엿볼 수 있다. 우리가 매일 사용하고 있는 익숙한 물건들에도 저마다가 가지고 있는 특별한 '무엇인가'가 있다. 일상적인 것에서 찾을 수 있는 소중한 이야기
6,7,9788979291452,그림 에세이,양태석 저,[시/에세이],[미술],[교양미술],191,2010-10-10,"『그림 에세이』의 저자 양태석이 평생 예술가의 길을 걸어오며 배운 것은 예술이 영혼을 살찌게 한다는 것이다. 예술에 심취하는 것은 마음을 밝히고 한 차원 높은 행복을 가져다준다. 예술은 영혼을 즐겁게하는 것이라는 그의 철학을 담아, 예술이 어려운 것이라고 생각하는 사람들을 위하여 이번 책을 출간하였다. 일반적으로 예술,"
7,8,9791196826444,"그러니 그냥 쓰다듬자, 마음을 - 당신의 도시를 보듬는 작은 위로",김지원 저/메그 그림,[시/에세이],[나라별 에세이],[한국에세이],160,2020-08-31,사람 냄새 진하게 풍기는 글을 읽다가 문득 그림에 시선이 멈춘다. 오래도록 그림에 마음을 맡기면 어느새 나도 모르게 다시 다음 글을 향해 간다. 나는 깨달았다. 위로는 일방적인 것이 아니라는 사실을. 작가는 오히려 나의 위로가 필요할 것 같은데 읽는 내내 더 큰 위로로 함께하며 나를 보듬어 주었다. 마음을 쓰다듬는다는 것
8,9,9788976227911,감성아크릴 - 언제 어디서나 누구나 쉽게 배우는,아가트 싱어 저,[예술/대중문화],[미술],[미술실기/기법],128,2019-01-11,* 어른이 되어서 즐기는 크레용 그림은 전문가가 아니더라도 자신감을 얻을 수 있는 새로운 종류의 예술이다. * 재미있고 간단한 테크닉으로 어디서나 즉흥적인 예술을 만들 수 있다. “그림 한 장 그려볼까?” 라는 생각을 하다가 이내 길을 잃은 기분이 들고 의욕을 잃거나 겁이 난 적이 있을 것이다. 그런 분들을 위해서 이 책
9,10,9788959253586,화음 - 수필가 100인선 89,박연구 저,[시/에세이],[나라별 에세이],[한국에세이],195,2011-08-10,"수필은 누구나 부담 없이 읽고, 마음만 먹으면 직접 쓸 수도 있는 가장 친근한 문학이다. 다른 영역의 문학이 영상매체에 밀려 신음하고 있는 도중에도 수필 인구만은 날로 증대하고 있다. 인생에 대한 깊이 있는 시선을 담은 잔잔한 에세이는 우리네 삶을 되돌아보게 하는 소중한 계기를 제공한다.삶에 대한 깊이 있는 관조를 담아"
